# Part II_Deep_Learning

## Group Members: Shishir Dahal, Royar Kurdman, Hashir Javed

## Model selection
We chose a 1-D CNN because the input is a raw vibration signal in which bearing faults appear as localized, repetitive impulses. Convolutional filters detect such local patterns independent of their position in the segment, and the network learns discriminative features directly from the waveform instead of relying on four hand-crafted statistics. Unlike an LSTM, a CNN trains efficiently on 2000-sample segments and has fewer parameters, which suits our small dataset of 384 segments.

## Task II.1 -- Engineering Problem and Dataset
### Question 1 
The Part I dataset is reused: vibration signals from the Paderborn University bearing test rig (bearings K001, KA01, KI01, condition N15_M01_F10). The problem is automatic bearing fault diagnosis. The target is a three-class label per segment: healthy (0), outer-race damage (1) or inner-race damage (2).
### Question 2 
The input is the raw vibration signal (vibration_1), not hand-crafted features. Each sample is one segment of 2000 consecutive values, giving a 1-D array of shape (384, 2000, 1): segments × time steps × channels. The sample size is 384 segments (128 per class).
### Question 3
A 1-D CNN is used because bearing faults appear as short, repeating impulses. Convolutional filters detect these local patterns at any position and learn features from the raw waveform.

## Task II.2 -- Data Preparation and Implementation
**Inputs and targets.** Every recording of `vibration_1` is cut into non-overlapping segments of 2000 samples (same segmentation as Part I). Each segment is one sample with shape (2000, 1); the target is the class label 0 = healthy (K001), 1 = outer race (KA01), 2 = inner race (KI01).

**Split without leakage.** Neighbouring segments of one recording are almost identical, so a segment-level random split would put near-copies of the same signal in train and test. We therefore split **by recording** (60 % / 20 % / 20 %, stratified per class, fixed seed). No recording contributes segments to more than one set. The standardisation constants (one mean and one standard deviation) are computed on the training set only and then applied to validation and test.

In [ ]:
import os, glob, re, json, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.io

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
keras.utils.set_random_seed(SEED)

DATA_DIR  = "dataset"     # folder with the extracted Paderborn .mat files (same as Part I)
SEG_LEN   = 2000
CONDITION = None          # None = all 4 operating conditions (as in Part I); e.g. "N15_M01_F10" for one condition
CLASSES = {"K001": 0, "KA01": 1, "KI01": 2}
NAMES   = {0: "Healthy", 1: "Outer race", 2: "Inner race"}
os.makedirs("figures", exist_ok=True)
print("TensorFlow", tf.__version__)

In [ ]:
def load_signal(path):
    """Return the vibration_1 signal of one .mat recording as a 1-D float array."""
    m = scipy.io.loadmat(path, squeeze_me=True, struct_as_record=False)
    key = [k for k in m if not k.startswith("__")][0]
    names = [y.Name for y in m[key].Y]
    return np.asarray(m[key].Y[names.index("vibration_1")].Data, float).ravel()

CACHE = f"segments_cache_{CONDITION or 'all'}.npz"      # speeds up re-runs; do NOT commit (see .gitignore)
if os.path.exists(CACHE):
    z = np.load(CACHE)
    X_all, y_all, rec_all = z["X"], z["y"], z["rec"]
else:
    X_list, y_list, rec_list = [], [], []
    for code_, label in CLASSES.items():
        files = sorted(glob.glob(os.path.join(DATA_DIR, "**", f"*{code_}*.mat"), recursive=True))
        if CONDITION:
            files = [f for f in files if CONDITION in os.path.basename(f)]
        print(f"{code_}: {len(files)} recordings")
        for path in files:
            sig = load_signal(path)
            n = len(sig) // SEG_LEN
            segs = sig[: n * SEG_LEN].reshape(n, SEG_LEN)
            segs = segs[segs.std(axis=1) > 0]                       # same rule as Part I
            X_list.append(segs.astype(np.float32))
            y_list.append(np.full(len(segs), label))
            rec_list.append(np.full(len(segs), os.path.basename(path)))
    if not X_list:
        raise FileNotFoundError(f"No .mat files found under {os.path.abspath(DATA_DIR)}")
    X_all, y_all, rec_all = np.concatenate(X_list), np.concatenate(y_list), np.concatenate(rec_list)
    np.savez_compressed(CACHE, X=X_all, y=y_all, rec=rec_all)

print("Segments:", X_all.shape, "| recordings:", len(np.unique(rec_all)))
print("Segments per class:", {NAMES[k]: int((y_all == k).sum()) for k in NAMES})

In [ ]:
# ---- split BY RECORDING, stratified per class (60/20/20) ----
rec_tbl = pd.DataFrame({"rec": rec_all, "y": y_all}).drop_duplicates("rec")
rng = np.random.default_rng(SEED)
split_of = {}
for lab in NAMES:
    recs = np.array(sorted(rec_tbl.loc[rec_tbl.y == lab, "rec"]))
    rng.shuffle(recs)
    n_tr, n_va = int(round(0.6 * len(recs))), int(round(0.2 * len(recs)))
    split_of.update({r: "train" for r in recs[:n_tr]})
    split_of.update({r: "val"   for r in recs[n_tr:n_tr + n_va]})
    split_of.update({r: "test"  for r in recs[n_tr + n_va:]})

split = pd.Series(rec_all).map(split_of).values
tr, va, te = split == "train", split == "val", split == "test"

# leakage check: no recording in more than one set
sets = [set(rec_all[m]) for m in (tr, va, te)]
assert not (sets[0] & sets[1] or sets[0] & sets[2] or sets[1] & sets[2]), "recording leakage!"

summary = pd.DataFrame({s: [len(set(rec_all[m])), int(m.sum())] + [int((y_all[m] == k).sum()) for k in NAMES]
                        for s, m in zip(["train", "val", "test"], [tr, va, te])},
                       index=["recordings", "segments"] + [f"segments {NAMES[k]}" for k in NAMES])
print(summary)

# ---- standardisation fitted on TRAIN only (one global mean/std) ----
mu, sd = X_all[tr].mean(), X_all[tr].std()
print(f"train mean = {mu:.4f}, train std = {sd:.4f}")

X_train, y_train, rec_train = ((X_all[tr] - mu) / sd)[..., None], y_all[tr], rec_all[tr]
X_val,   y_val,   rec_val   = ((X_all[va] - mu) / sd)[..., None], y_all[va], rec_all[va]
X_test,  y_test,  rec_test  = ((X_all[te] - mu) / sd)[..., None], y_all[te], rec_all[te]
X_test_raw = X_all[te].copy()          # un-scaled copy, used later for plots and feature baseline
X_train_raw = X_all[tr].copy()
del X_all
print("X_train", X_train.shape, "| X_val", X_val.shape, "| X_test", X_test.shape)

### CNN architecture
The model is a 1-D CNN with two convolutional blocks, global average pooling and a softmax output layer.

| Layer | Output shape | Settings | Trainable parameters |
|---|---|---|---|
| Input | (2000, 1) | one vibration segment, standardised | 0 |
| Conv1D | (2000, 16) | 16 filters, kernel size 32, ReLU, `same` padding | 528 |
| MaxPooling1D | (500, 16) | pool size 4 | 0 |
| Conv1D | (500, 32) | 32 filters, kernel size 16, ReLU, `same` padding | 8 224 |
| MaxPooling1D | (125, 32) | pool size 4 | 0 |
| GlobalAveragePooling1D | (32,) | averages each filter over time | 0 |
| Dropout | (32,) | rate 0.3 (regularisation) | 0 |
| Dense | (3,) | softmax, one unit per class | 99 |
| **Total** | | | **8 851** |

**Main components.** The convolutional filters are learned detectors for short impulse patterns (the first, longer kernel covers 32 samples and works like a learned band-pass filter; the second, shorter kernel combines these into higher-level patterns). Max pooling reduces the sequence length and gives some shift tolerance. Global average pooling aggregates each filter over the whole segment, so the output does not depend on where in the segment an impulse occurs and the network needs no large dense layer. Dropout and the small parameter count limit overfitting.

In [ ]:
def build_cnn(filters=(16, 32), kernels=(32, 16), pool=4, dropout=0.3, seed=SEED):
    keras.utils.set_random_seed(seed)
    return keras.Sequential([
        keras.Input(shape=(SEG_LEN, 1)),
        layers.Conv1D(filters[0], kernels[0], padding="same", activation="relu"),
        layers.MaxPooling1D(pool),
        layers.Conv1D(filters[1], kernels[1], padding="same", activation="relu"),
        layers.MaxPooling1D(pool),
        layers.GlobalAveragePooling1D(),
        layers.Dropout(dropout),
        layers.Dense(len(NAMES), activation="softmax"),
    ], name="cnn1d_bearing")

model = build_cnn()
model.summary()
n_params = model.count_params()
print("Trainable parameters:", sum(int(np.prod(w.shape)) for w in model.trainable_weights), "| total:", n_params)

## Task II.3 -- Training and Evaluation
**Optimizer and loss.** Adam (learning rate 1e-3) with sparse categorical cross-entropy, which is the standard choice for integer-coded multiclass labels with a softmax output.

**Epochs and stopping rule.** Maximum **30 epochs**, batch size 128. Training stops early when the validation loss has not improved for **5 consecutive epochs**, and the weights of the epoch with the lowest validation loss are restored. The test set is not used for any of these decisions.

In [ ]:
MAX_EPOCHS, PATIENCE, BATCH = 30, 5, 128

model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
callbacks = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE,
                                           restore_best_weights=True)]
history = model.fit(X_train[::3], y_train[::3],
                    validation_data=(X_val, y_val),
                    epochs=MAX_EPOCHS, batch_size=BATCH,
                    callbacks=callbacks, verbose=2)
hist = pd.DataFrame(history.history)
best_epoch = int(hist["val_loss"].idxmin()) + 1
print(f"Stopped after {len(hist)} epochs; best epoch (lowest val loss) = {best_epoch}")

In [ ]:
# ---- learning curves ----
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ep = np.arange(1, len(hist) + 1)
ax[0].plot(ep, hist["loss"], label="train"); ax[0].plot(ep, hist["val_loss"], label="validation")
ax[0].axvline(best_epoch, ls="--", c="gray", label=f"best epoch ({best_epoch})")
ax[0].set_xlabel("Epoch"); ax[0].set_ylabel("Cross-entropy loss"); ax[0].set_title("Loss"); ax[0].legend()
ax[1].plot(ep, hist["accuracy"], label="train"); ax[1].plot(ep, hist["val_accuracy"], label="validation")
ax[1].axvline(best_epoch, ls="--", c="gray")
ax[1].set_xlabel("Epoch"); ax[1].set_ylabel("Accuracy"); ax[1].set_title("Accuracy"); ax[1].legend()
plt.tight_layout(); plt.savefig("figures/fig_ii3_learning_curves.png", dpi=150); plt.show()

In [ ]:
# ---- evaluation on the held-out TEST set (recordings never seen in training/validation) ----
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, precision_recall_fscore_support)

proba  = model.predict(X_test, batch_size=256, verbose=0)
y_pred = proba.argmax(axis=1)

acc = accuracy_score(y_test, y_pred)
p, r, f1, _ = precision_recall_fscore_support(y_test, y_pred, average="macro", zero_division=0)
print(f"Test accuracy: {acc:.4f}")
print(f"Macro-averaged precision {p:.4f} | recall {r:.4f} | F1 {f1:.4f}  (macro = unweighted mean over the 3 classes)\n")
print(classification_report(y_test, y_pred, target_names=list(NAMES.values()), digits=4, zero_division=0))

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(4.8, 4.2))
ConfusionMatrixDisplay(cm, display_labels=list(NAMES.values())).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("CNN -- test confusion matrix"); plt.tight_layout()
plt.savefig("figures/fig_ii3_confusion_matrix.png", dpi=150); plt.show()

train_acc_final = float(model.evaluate(X_train, y_train, verbose=0)[1])
val_acc_final   = float(model.evaluate(X_val,   y_val,   verbose=0)[1])
print(f"Accuracy  train {train_acc_final:.4f} | validation {val_acc_final:.4f} | test {acc:.4f}")

### Reference: feature-based baseline on the *same* split
To judge whether the CNN is meaningfully better, an RBF-SVM on the four hand-crafted features of Part I (RMS, peak, crest factor, kurtosis) is trained on exactly the same training recordings and scored on exactly the same test segments. (Hyperparameters are the defaults; the tuned SVM from Part I Task I.4 was evaluated on a different, segment-level split, so its numbers are not directly comparable.)

In [ ]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score

def features(X):
    rms  = np.sqrt((X ** 2).mean(axis=1))
    peak = np.abs(X).max(axis=1)
    z    = (X - X.mean(axis=1, keepdims=True)) / X.std(axis=1, keepdims=True)
    return np.column_stack([rms, peak, peak / rms, (z ** 4).mean(axis=1)])

F_train, F_test = features(X_train_raw), features(X_test_raw)
sc = StandardScaler().fit(F_train)                                    # fitted on train only
svm = SVC(kernel="rbf", C=1.0, gamma="scale", random_state=SEED).fit(sc.transform(F_train), y_train)
svm_pred = svm.predict(sc.transform(F_test))

comparison = pd.DataFrame({
    "accuracy":  [accuracy_score(y_test, svm_pred), acc],
    "macro F1":  [f1_score(y_test, svm_pred, average="macro"), f1],
}, index=["SVM-RBF (4 features)", "1-D CNN (raw signal)"]).round(4)
print(comparison)

os.makedirs("results", exist_ok=True)
json.dump({"test_accuracy": acc, "macro_precision": p, "macro_recall": r, "macro_f1": f1,
           "svm_baseline_accuracy": float(accuracy_score(y_test, svm_pred)),
           "epochs_run": len(hist), "best_epoch": best_epoch, "n_params": int(n_params)},
          open("results/ii3_metrics.json", "w"), indent=2)

### Observations (complete after running -- replace the bracketed values with your output)
- Learning curves: training and validation loss [both decrease / diverge after epoch X]. The best epoch was [X] of [Y] epochs run, so the stopping rule [was / was not] triggered before the 30-epoch maximum.
- Test performance: accuracy [..], macro precision [..], macro recall [..], macro F1 [..] on [N] test segments from [R] unseen recordings.
- Compared with the RBF-SVM on hand-crafted features on the same split ([..] accuracy), the CNN is [better / similar / worse] by [..] percentage points.
- The gap between validation accuracy ([..]) and test accuracy ([..]) [is small / shows ...].

## Task II.4 -- Model Behavior (Option B: Prediction Errors)
We look at which classes, operating conditions and recordings the CNN predicts poorly, and at what the misclassified signals look like.

In [ ]:
res = pd.DataFrame({"rec": rec_test, "true": y_test, "pred": y_pred, "conf": proba.max(axis=1)})
res["correct"] = res.true == res.pred
res["condition"] = res.rec.str.extract(r"(N\d+_M\d+_F\d+)")[0]      # speed_torque_force code in the file name

print("Errors:", int((~res.correct).sum()), "of", len(res), f"({(~res.correct).mean():.2%})\n")
print("Confusion counts (rows = true, columns = predicted):")
print(pd.crosstab(res.true.map(NAMES), res.pred.map(NAMES)))

print("\nAccuracy by class:");               print(res.groupby(res.true.map(NAMES)).correct.mean().round(4))
print("\nAccuracy by operating condition:"); print(res.groupby("condition").correct.mean().round(4))

pivot = res.pivot_table(index="condition", columns=res.true.map(NAMES), values="correct", aggfunc="mean")
print("\nAccuracy by condition and class:"); print(pivot.round(3))

rec_acc = res.groupby(["rec", res.true.map(NAMES)]).correct.mean().rename("accuracy").reset_index().sort_values("accuracy")
print("\nWorst 8 test recordings:"); print(rec_acc.head(8).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
pivot.plot(kind="bar", ax=ax[0], ylim=(0, 1.05)); ax[0].set_ylabel("Test accuracy"); ax[0].set_xlabel("Operating condition")
ax[0].set_title("Accuracy by operating condition and class"); ax[0].tick_params(axis="x", rotation=30)
ax[1].hist(res.loc[res.correct, "conf"], bins=20, alpha=0.7, label="correct")
if (~res.correct).any():
    ax[1].hist(res.loc[~res.correct, "conf"], bins=20, alpha=0.7, label="wrong")
ax[1].set_xlabel("Predicted probability of chosen class"); ax[1].set_ylabel("Segments"); ax[1].set_yscale("log")
ax[1].set_title("Model confidence"); ax[1].legend()
plt.tight_layout(); plt.savefig("figures/fig_ii4_errors_by_condition.png", dpi=150); plt.show()

# ---- what do the misclassified signals look like? ----
wrong_idx = np.where(~res.correct.values)[0]
if len(wrong_idx):
    pick = wrong_idx[np.argsort(-res.conf.values[wrong_idx])][:3]          # the 3 most confident mistakes
    ok_idx = [np.where((res.true.values == res.true.values[i]) & res.correct.values)[0][0] for i in pick]
    fig, axes = plt.subplots(2, len(pick), figsize=(5 * len(pick), 5), sharey=True, squeeze=False)
    for j, (i, k) in enumerate(zip(pick, ok_idx)):
        axes[0, j].plot(X_test_raw[i], lw=0.6)
        axes[0, j].set_title(f"WRONG: true {NAMES[res.true[i]]} -> pred {NAMES[res.pred[i]]} ({res.conf[i]:.2f})\n{res.condition[i]}", fontsize=9)
        axes[1, j].plot(X_test_raw[k], lw=0.6, c="tab:green")
        axes[1, j].set_title(f"CORRECT example of {NAMES[res.true[i]]} ({res.condition[k]})", fontsize=9)
        axes[1, j].set_xlabel("Sample")
    axes[0, 0].set_ylabel("Vibration"); axes[1, 0].set_ylabel("Vibration")
    plt.tight_layout(); plt.savefig("figures/fig_ii4_misclassified_examples.png", dpi=150); plt.show()

    # signal statistics: are the mistakes statistically "unusual" for their true class?
    F = pd.DataFrame(features(X_test_raw), columns=["rms", "peak", "crest", "kurtosis"])
    F["true"] = res.true.map(NAMES).values; F["status"] = np.where(res.correct, "correct", "wrong")
    print(F.groupby(["true", "status"]).median().round(3))
else:
    print("No test errors -- see the discussion about why a perfect score needs care (single bearing per class).")

### Discussion (complete after running -- use your own numbers)
**Where the errors occur.** [Name the class pair(s) in the confusion matrix with the most errors, with counts, and the operating condition(s)/recordings with the lowest accuracy.]

**Possible engineering explanations (hypotheses to check against the output above).**
1. *Operating condition.* If errors concentrate in conditions with low rotational speed or low load/radial force (e.g. N09_M07_F10 or N15_M07_F04), the fault impacts are weaker and the damaged signal resembles the healthy one more closely, since impact energy grows with speed and load.
2. *Fault type.* Inner-race faults pass through a varying load zone and give amplitude-modulated impulses, while outer-race impulses are more regular. A 2000-sample segment (about 31 ms at 64 kHz) may contain only one or two impacts, so some segments of a faulty bearing contain no visible impact at all.
3. *Confident mistakes.* If the wrong predictions have high confidence, the signal statistics of those segments (table above) differ from the typical segments of their class, which points to recording-specific effects rather than model uncertainty.
4. *Dataset limits.* Each class is one physical bearing with artificial damage, so the network may partly learn bearing-specific vibration signatures; errors on whole recordings would then indicate a recording that differs from the training recordings of that bearing.

**Conclusion.** [Two or three sentences: what the evidence supports and what experiment (e.g. leave-one-condition-out testing) would separate the explanations.]

# Individual Analysis -- Hashir Javed

## Question II.1 -- Architecture Selection
*(write your own answer here, max 300 words)*

## Question II.2 -- Model Performance
*(write your own answer here, max 300 words, with numbers from Task II.3)*

## Question II.3 -- Engineering Applicability
*(write your own answer here, max 300 words)*